In [ ]:
import h5py
import torch
import torch.nn.functional as F
import fastmri
from fastmri.data import transforms as T
import matplotlib.pyplot as plt
import numpy as np
import os



# ──────────────────────────────────────────────
# 1. Core utility: crop OR pad complex tensors
# ──────────────────────────────────────────────
def complex_center_crop_or_pad(data, target_shape):
    """
    Center-crop or zero-pad complex data to target spatial size.

    Args:
        data:         (..., H, W, 2)   — last dim is [real, imag]
        target_shape: (target_H, target_W)

    Returns:
        (..., target_H, target_W, 2)
    """
    h, w = data.shape[-3], data.shape[-2]
    th, tw = target_shape

    if h < th:                        # pad
        pad_top = (th - h) // 2
        pad_bot = th - h - pad_top
        # F.pad order: (last_L, last_R, second_last_L, second_last_R, third_last_L, third_last_R)
        data = F.pad(data, (0, 0, 0, 0, pad_top, pad_bot))

    w = data.shape[-2]  
    if w < tw:                        # pad
        pad_left  = (tw - w) // 2
        pad_right = tw - w - pad_left
        data = F.pad(data, (0, 0, pad_left, pad_right))


    if w > tw or h>th:                          # crop
        data = T.complex_center_crop(data=data,shape=target_shape)


    return data

def tensor_to_complex_np(data):
    """
    (..., H, W, 2) real tensor  →  (..., H, W) complex64 numpy
    """
    return torch.view_as_complex(data.contiguous()).numpy()

#processing pipeline
target_path= "/home/biswamitra/health/knee_data/train/deconstructed_train"
target_dir = os.listdir(target_path)

for file_names in target_dir:
    final_path = os.path.join(target_path,file_names)
    #logical process : load -> convert to tensor -> ifft2c -> crop or pad -> fft2c -> save
    data = np.load(final_path)
    data_tensor = T.to_tensor(data)
    data_ifft = fastmri.ifft2c(data_tensor)
    process_data = complex_center_crop_or_pad(data_ifft,(640,368))
    fft_data = fastmri.fft2c(process_data)
    complex_np = tensor_to_complex_np(fft_data) 
    np.save(final_path,complex_np)